# COde 03 — Đánh giá, so sánh và xuất Student

Chạy sau notebook 02. Mặc định đánh giá validation để kiểm tra; chuyển sang test sau khi khóa protocol.
Không huấn luyện hay chọn checkpoint trong notebook này.
Mỗi lần đánh giá kiểm tra release và source trùng phiên bản huấn luyện.

Nếu đánh giá Teacher trên test, trước đó cần thêm shard test của **cùng encoder cache** bằng mục 14 notebook 01.
Student KD không cần clinical cache để suy luận. Bảng so sánh toàn bộ suite gồm cả Teachers nên cần cache cho các hàng Teacher.

## Thiết lập offline trước khi import thư viện

Trên Kaggle, gắn bundle tạo bởi notebook **00**. Cell sau cài bằng `--no-index`, kiểm tra phiên bản và dùng weights local.
Nếu đang chủ động chạy phiên online chưa có bundle, đặt `OFFLINE_MODE=False`; chế độ GPU cuộc thi dùng True.

In [ ]:
# Chạy trước khi import torch/transformers/pandas.
from pathlib import Path
import os, sys, importlib.util
OFFLINE_MODE = Path('/kaggle').exists()
OFFLINE_BUNDLE_OVERRIDE = None  # folder chứa offline_manifest.json
OFFLINE_ASSETS = None
if OFFLINE_MODE:
    candidates = []
    for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            here = Path(current)
            if 'offline_manifest.json' in files:
                candidates.append(here)
                dirs[:] = []
                continue
            dirs[:] = [d for d in dirs if d not in {'Images', '.cache', '__pycache__', 'runs', 'models', 'wheels'}]
            if len(here.relative_to(base).parts) >= 7:
                dirs[:] = []
    if OFFLINE_BUNDLE_OVERRIDE is not None:
        bundle = Path(OFFLINE_BUNDLE_OVERRIDE)
    elif len(candidates) == 1:
        bundle = candidates[0]
    else:
        raise RuntimeError(f'Attach offline bundle prepared by notebook 00; found: {candidates}. Set OFFLINE_BUNDLE_OVERRIDE.')
    helper = bundle / 'source' / '_support' / 'kaggle_offline.py'
    spec = importlib.util.spec_from_file_location('kaggle_offline', helper)
    offline = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(offline)
    OFFLINE_ASSETS = offline.activate_bundle(bundle)
    sys.path.insert(0, str(bundle / 'source'))
    print('Offline setup OK. Local pretrained weights:', OFFLINE_ASSETS['resnet'])
else:
    print('Local/online mode: no offline installation. Kaggle offline uses notebook 00 bundle.')

In [ ]:
from pathlib import Path
import os, sys, json

PROJECT_ROOT_OVERRIDE = None  # folder chứa clindistill/ + configs/
PREPARED_ROOT_OVERRIDE = None  # folder chứa release.json
IMAGE_ROOT_OVERRIDE = None  # folder chứa Images/Photographs
CLINICAL_CACHE_OVERRIDE = None  # folder chứa metadata.json + train.npz + validation.npz

def find_roots(predicate):
    results = set()
    bases = [Path.cwd(), Path.cwd().parent / 'Data', Path('/kaggle/input')]
    for base in bases:
        if not base.exists():
            continue
        for current, dirs, files in os.walk(base):
            here = Path(current)
            if predicate(here, dirs, files):
                results.add(here.resolve())
            dirs[:] = [d for d in dirs if d not in {'Images', '.git', '.cache', '__pycache__',
                'node_modules', 'runs', 'outputs', '_support', '_archive'}]
            if len(here.relative_to(base).parts) >= 7:
                dirs[:] = []
    return sorted(results, key=str)

def resolve_one(override, candidates, kind, optional=False):
    if override is not None:
        path = Path(override).resolve()
        if not path.is_dir():
            raise FileNotFoundError(f'{kind}: {path}')
        return path
    if optional and not candidates:
        return None
    if len(candidates) != 1:
        raise RuntimeError(f'{kind}: found {len(candidates)} folders: {candidates}. Set the corresponding *_OVERRIDE.')
    return candidates[0]

PROJECT_ROOT = resolve_one(PROJECT_ROOT_OVERRIDE,
    [OFFLINE_ASSETS['root'] / 'source'] if globals().get('OFFLINE_ASSETS') else
    [Path.cwd()] if (Path.cwd() / 'clindistill').is_dir() else
    find_roots(lambda p, dirs, files: 'clindistill' in dirs and 'configs' in dirs), 'Project source')
sys.path.insert(0, str(PROJECT_ROOT))
PREPARED_ROOT = resolve_one(PREPARED_ROOT_OVERRIDE,
    find_roots(lambda p, dirs, files: 'release.json' in files and
               not json.loads((p / 'release.json').read_text(encoding='utf-8')).get('debug_subset')), 'Prepared release')
IMAGE_ROOT = resolve_one(IMAGE_ROOT_OVERRIDE,
    find_roots(lambda p, dirs, files: 'Images' in dirs and (p / 'Images' / 'Photographs').is_dir()
               and 'mini_dataset.json' not in files), 'COde images')
CLINICAL_CACHE = resolve_one(CLINICAL_CACHE_OVERRIDE,
    find_roots(lambda p, dirs, files: {'metadata.json', 'train.npz', 'validation.npz'} <= set(files) and
        json.loads((p / 'metadata.json').read_text(encoding='utf-8'))['request']['release_id'] ==
        json.loads((PREPARED_ROOT / 'release.json').read_text(encoding='utf-8'))['release_id']),
    'Clinical cache', optional=True)
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else PROJECT_ROOT
print('Source:', PROJECT_ROOT, '\nPrepared:', PREPARED_ROOT, '\nImages:', IMAGE_ROOT,
      '\nClinical cache:', CLINICAL_CACHE, '\nOutput:', OUTPUT_ROOT)

In [ ]:
from clindistill.reporting import evaluate_suite, export_student, predict_images
from clindistill.artifacts import read_json

SMOKE_TEST = os.environ.get('CLINDISTILL_SMOKE_TEST') == '1'
RUN_NAME = 'code_clindistill_v2'
RUNS_ROOT_OVERRIDE = None  # folder run đã COPY về /kaggle/working nếu từ phiên trước
RUNS_ROOT = Path(RUNS_ROOT_OVERRIDE) if RUNS_ROOT_OVERRIDE else (
    OUTPUT_ROOT / '_support' / 'smoke_runs' / RUN_NAME if SMOKE_TEST else OUTPUT_ROOT / 'runs' / RUN_NAME)
EVALUATION_SPLIT = 'validation'
ALLOW_EXPERIMENTAL_TEST = False
RUN_EVALUATION = True
print('Runs:', RUNS_ROOT, '| partition:', EVALUATION_SPLIT)

In [ ]:
per_seed = summary = comparisons = None
if RUN_EVALUATION:
    per_seed, summary, comparisons = evaluate_suite(PREPARED_ROOT, IMAGE_ROOT, RUNS_ROOT,
        split=EVALUATION_SPLIT, allow_experimental_test=ALLOW_EXPERIMENTAL_TEST,
        clinical_cache=CLINICAL_CACHE)
    display(per_seed)
    display(summary)
    print(json.dumps(comparisons, indent=2, ensure_ascii=False))

## 1. So sánh nhiều seed

Error bar dưới đây là **sample std giữa seed**, không phải confidence interval.
CI trong từng run bootstrap theo split_group; paired CI dùng cùng case/group/seed giữa hai mô hình.
Các CI này có điều kiện trên mô hình đã fit, không thay thế biến thiên do train lại.
Nếu chỉ một seed, std không xác định; không trình bày là độ lệch chuẩn bằng 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
if per_seed is not None:
    report_dir = RUNS_ROOT / f'report_{EVALUATION_SPLIT}'
    agg = per_seed.groupby('experiment').macro_f1.agg(['mean', 'std', 'count'])
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(agg.index, agg['mean'], color='#247ba0')
    valid = agg['count'] > 1
    ax.errorbar(np.flatnonzero(valid), agg.loc[valid, 'mean'], yerr=agg.loc[valid, 'std'],
                fmt='none', capsize=3, color='black')
    ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=65)
    ax.set_title(f'{EVALUATION_SPLIT}: mean and seed std' + (' [DEBUG]' if SMOKE_TEST else ''))
    plt.tight_layout(); plt.savefig(report_dir / 'comparison.png', dpi=160); plt.show()

## 2. Confusion matrix và đường risk–coverage của một Student

Risk–coverage xếp ca theo max probability, báo tỷ lệ sai khi chỉ giữ ca tự tin nhất.
Nó không chứng minh mô hình biết khi nào cần khám lâm sàng. Attention theo ảnh không phải định vị tổn thương.
Các file dự đoán có case ID để đối chiếu lỗi với dữ liệu gốc; không nhúng hồ sơ bệnh nhân vào artifact inference.

In [ ]:
import pandas as pd
INSPECT_EXPERIMENT = 'kd_selective'
INSPECT_SEED = 2026
run_dir = RUNS_ROOT / INSPECT_EXPERIMENT / f'seed_{INSPECT_SEED}'
metrics_file = run_dir / f'evaluation_{EVALUATION_SPLIT}' / 'metrics.json'
if metrics_file.exists():
    result = read_json(metrics_file)
    classes = read_json(run_dir / 'run_config.json')['class_names']
    matrix = np.asarray(result['metrics']['confusion_matrix'])
    curve = pd.read_csv(run_dir / f'evaluation_{EVALUATION_SPLIT}' / 'risk_coverage.csv')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].imshow(matrix, cmap='Blues')
    axes[0].set_xticks(range(len(classes)), classes, rotation=65, ha='right')
    axes[0].set_yticks(range(len(classes)), classes)
    axes[0].set_xlabel('Prediction'); axes[0].set_ylabel('Recorded target')
    for i in range(len(classes)):
        for j in range(len(classes)):
            axes[0].text(j, i, str(matrix[i, j]), ha='center', va='center')
    axes[1].plot(curve.coverage, curve.risk)
    axes[1].set(xlabel='Coverage', ylabel='Error risk', xlim=(0, 1), ylim=(0, 1))
    plt.tight_layout(); plt.savefig(run_dir / f'evaluation_{EVALUATION_SPLIT}' / 'diagnostics.png', dpi=160); plt.show()
else:
    print('Chưa có nhánh này. Chọn INSPECT_EXPERIMENT/SEED đã train.')

## 3. Xuất Student dùng ảnh, không cần hồ sơ

Chọn checkpoint bằng validation trước khi mở test. `student.pt` chỉ có weights và cấu hình, không có optimizer hoặc clinical data.
Dùng cùng source clindistill với artifact; chỉ tải file .pt do bạn tạo/tin cậy.
Inference trả phân phối trên 6 nhãn nghiên cứu, không có lớp healthy và không dùng như kết luận khám bệnh.

In [ ]:
EXPORT_STUDENT = False  # bật sau khi chọn Student bằng validation
if EXPORT_STUDENT:
    artifact = export_student(run_dir, OUTPUT_ROOT / 'exports' / RUN_NAME / 'student.pt')
    print('Exported:', artifact)
    # Ví dụ; thay bằng đường dẫn ảnh thật, tương đối với IMAGE_ROOT:
    # result = predict_images(artifact, IMAGE_ROOT, ['Images/Photographs/example.jpg'])
    # print(result)

Output tổng hợp: `report_<split>/per_seed.csv`, `mean_std.csv`, `paired_comparisons.json`, `comparison.png`, `status.json`.
Từng run có per-class recall/AUROC, ECE, Brier, confusion matrix và CI.

**Phần cần làm sau khi có kết quả:** kiểm tra ảnh gần trùng/clinical leakage, ablation trọng số và representation,
phân tích lỗi theo hỗ trợ từng lớp, chốt kết luận dựa trên kết quả thật. MeMoSA cần dữ liệu và mapping nhãn độc lập;
không áp classifier 6 lớp COde để tuyên bố external validation.